<a href="https://colab.research.google.com/github/Davidsudheerkumar/24P31A4255_DriveReady_OPC_2026/blob/main/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:

!pip install -q -U langchain langchain-groq gspread pydantic

import os
from google.colab import userdata, auth

# Get Groq API key from Colab Secrets
api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

os.environ["GROQ_API_KEY"] = api_key

from langchain_groq import ChatGroq

# Initialize Groq model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

# Test the model
response = llm.invoke("Say EduPredictAI is ready.")
print(response.content)

# Google Sheets authentication
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Create spreadsheet (run only once)
sh = gc.create("EduPredictAI Student Records")
ws = sh.sheet1

ws.append_row([
    "name",
    "subject",
    "marks",
    "attendance",
    "date",
    "concern",
    "performance"
])

print("Google Sheet ready:", sh.url)


EduPredictAI is ready.
Google Sheet ready: https://docs.google.com/spreadsheets/d/1hax3ih1vyq6G9FNZ7myLRRb3Ya2PsqjlRSA1Hoyl48Y


In [36]:

from typing import Optional
from pydantic import BaseModel, Field
from langchain.tools import tool

# Student performance rules
def classify_score(score):
    if score < 0 or score > 20:
        return "Invalid score"
    elif score == 0:
        return "Dropout"
    elif score <= 9:
        return "Fail"
    else:
        return "Pass"


# Student record structure
class StudentRecord(BaseModel):
    """A student academic record extracted from a message."""

    name: str = Field(description="Student's full name")
    subject: str = Field(description="Subject name")
    marks: float = Field(description="Marks obtained")
    attendance: float = Field(
        ge=0,
        le=100,
        description="Attendance percentage"
    )
    date: str = Field(description="Date of the record")
    concern: Optional[str] = Field(
        default=None,
        description="Student's academic concern"
    )


# Tool 1: Calculate percentage
@tool
def calculate_percentage(marks_obtained: float, total_marks: float) -> str:
    """Calculate a student's marks percentage.

    Args:
        marks_obtained: Marks obtained by the student.
        total_marks: Maximum possible marks.
    """

    if total_marks <= 0:
        return "Total marks must be greater than zero."

    if marks_obtained < 0 or marks_obtained > total_marks:
        return "Invalid marks. Check the marks entered."

    percentage = (marks_obtained / total_marks) * 100

    return f"Student percentage is {percentage:.2f}%."


# Tool 2: Classify student performance
@tool
def classify_performance(score: int) -> str:
    """Classify a student's performance using a score from 0 to 20.

    Args:
        score: Student's final grade between 0 and 20.
    """

    return classify_score(score)


# Tool 3: Save student record
@tool
def log_student_record(
    name: str,
    subject: str,
    marks: float,
    attendance: float,
    date: str,
    concern: str = "Not provided"
) -> str:
    """Save a student's academic record to the EduPredictAI Google Sheet.

    Use only when the student record details have been confirmed.
    """

    try:
        record = StudentRecord(
            name=name,
            subject=subject,
            marks=marks,
            attendance=attendance,
            date=date,
            concern=concern
        )

        performance = classify_score(int(record.marks))

        ws.append_row([
            record.name,
            record.subject,
            record.marks,
            record.attendance,
            record.date,
            record.concern or "Not provided",
            performance
        ])

        return (
            f"Record saved successfully for {record.name}. "
            f"Performance: {performance}."
        )

    except Exception as e:
        return f"Could not save record: {e}"


print("EduPredictAI tools created successfully!")


EduPredictAI tools created successfully!


In [37]:

structured_llm = llm.with_structured_output(StudentRecord)

messages = [
    "Ravi scored 78 marks in Mathematics. Attendance 90%. Date 26 November 2026. Needs exam preparation.",
    "Priya got 45 marks in Physics, attendance 75%, on 26 November 2026. She needs help with numerical problems.",
    "My name is Arun. I scored 15 in Computer Science. Attendance is 85%. Date: 26 November 2026.",
    "Sneha scored 88 in English. Her attendance is 95%. Date: 26 November 2026. No concerns.",
    "Rahul got 32 marks in Chemistry. Attendance 60%. Date 26 November 2026. He needs extra academic support."
]

for i, message in enumerate(messages, start=1):
    try:
        record = structured_llm.invoke(
            "Extract the student details from this message. "
            "Do not invent missing information. "
            "Use 'Not provided' for missing required text fields "
            "and 0 for missing numeric fields. "
            "Message: " + message
        )

        print(f"\nStudent Record {i}")
        print(record.model_dump())

    except Exception as e:
        print(f"Error in record {i}:", e)



Student Record 1
{'name': 'Ravi', 'subject': 'Mathematics', 'marks': 78.0, 'attendance': 90.0, 'date': '26 November 2026', 'concern': 'Needs exam preparation.'}

Student Record 2
{'name': 'Priya', 'subject': 'Physics', 'marks': 45.0, 'attendance': 75.0, 'date': '26 November 2026', 'concern': 'She needs help with numerical problems.'}

Student Record 3
{'name': 'Arun', 'subject': 'Computer Science', 'marks': 15.0, 'attendance': 85.0, 'date': '26 November 2026', 'concern': None}

Student Record 4
{'name': 'Sneha', 'subject': 'English', 'marks': 88.0, 'attendance': 95.0, 'date': '26 November 2026', 'concern': 'No concerns.'}

Student Record 5
{'name': 'Rahul', 'subject': 'Chemistry', 'marks': 32.0, 'attendance': 60.0, 'date': '26 November 2026', 'concern': 'He needs extra academic support.'}


In [38]:

from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are EduPredictAI, an AI-powered student academic support assistant.

FACTS:
EduPredictAI helps teachers understand student academic performance.

RULES:
1. Always use calculate_percentage for percentage calculations.
2. Always use classify_performance for score classification.
3. Use log_student_record only when the student has confirmed the details.
4. Never invent student marks or attendance.
5. Ask for missing information before saving a record.
6. Explain results in simple English.
7. Never claim that a record was saved unless the tool succeeds.

PERFORMANCE:
0 = Dropout
1 to 9 = Fail
10 to 20 = Pass
"""

agent = create_agent(
    model=llm,
    tools=[
        calculate_percentage,
        classify_performance,
        log_student_record
    ],
    system_prompt=SYSTEM
)

history = []

def chat(text):
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", str(e))
        return "Sorry, something went wrong. Please try again."


print(chat(
    "A student scored 7 out of 20. "
    "Calculate the percentage and classify the performance."
))


The student’s score of **7 out of 20** works out to a **percentage of 35 %**.  
According to our performance scale, a score of 7 falls into the **“Fail”** category.


In [39]:

print(chat("A student scored 15 out of 20. What is the percentage and result?"))

print(chat("Calculate the percentage for 450 marks out of 500."))

print(chat("Classify a student who scored 0 out of 20."))


The student scored **15 out of 20**.

1. **Percentage**  
   \[
   \frac{15}{20} \times 100 = 75\%
   \]

2. **Result**  
   Using our performance scale, a score of 15 is in the **“Pass”** range (10‑20).  

So the student has a **75 %** score and has **passed**.
The percentage is **90 %**.  

Calculation: \(\frac{450}{500} \times 100 = 90\%\).
A score of **0 out of 20** means the student is classified as **“Dropout.”**


In [40]:

print(chat(
    "Save this confirmed student record: "
    "Name Ravi, subject Mathematics, marks 78, "
    "attendance 90 percent, date 26 November 2026, "
    "concern exam preparation."
))

print("Open your Google Sheet:", sh.url)


I’m ready to save the record, but I need one more detail to make sure everything is correct:

**What was the maximum possible marks for the Mathematics exam?** (You mentioned the student scored 78, but I need the total marks to calculate the percentage and classify the performance.)
Open your Google Sheet: https://docs.google.com/spreadsheets/d/1hax3ih1vyq6G9FNZ7myLRRb3Ya2PsqjlRSA1Hoyl48Y


In [41]:

@tool("tool1")
def calculate_percentage_bad(
    marks_obtained: float,
    total_marks: float
) -> str:
    """does stuff"""

    if total_marks <= 0:
        return "Invalid total marks."

    return str((marks_obtained / total_marks) * 100)


bad_agent = create_agent(
    model=llm,
    tools=[calculate_percentage_bad, classify_performance],
    system_prompt="You are a student support assistant. Use tools when needed."
)

result = bad_agent.invoke({
    "messages": [
        {"role": "user", "content": "Calculate 450 out of 500."}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Calculate 450 out of 500.
================================== Ai Message ==================================

**450 out of 500** is **90 %**.

If you’d like to see what that would be on a 0‑20 scale:

\[
\text{Score on 0–20} = \frac{450}{500}\times 20 = 18
\]

So you’d have an 18/20, which is typically considered “Excellent.”


In [42]:

trace_result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Calculate the percentage for 450 out of 500 and classify score 15."
    }]
})

tool_calls = 0
model_calls = 0

for msg in trace_result["messages"]:
    msg.pretty_print()

    if msg.type == "ai":
        model_calls += 1
        tool_calls += len(getattr(msg, "tool_calls", []))

print("\nTotal model calls:", model_calls)
print("Total tool calls:", tool_calls)


================================ Human Message =================================

Calculate the percentage for 450 out of 500 and classify score 15.
================================== Ai Message ==================================
Tool Calls:
  calculate_percentage (fc_004640b0-0d2c-4afe-ae72-d682da4d0031)
 Call ID: fc_004640b0-0d2c-4afe-ae72-d682da4d0031
  Args:
    marks_obtained: 450
    total_marks: 500
================================= Tool Message =================================
Name: calculate_percentage

Student percentage is 90.00%.
================================== Ai Message ==================================
Tool Calls:
  classify_performance (fc_ba5b22c1-b553-4205-a852-6a8cdc2f555c)
 Call ID: fc_ba5b22c1-b553-4205-a852-6a8cdc2f555c
  Args:
    score: 15
================================= Tool Message =================================
Name: classify_performance

Pass
================================== Ai Message ==================================

Here’s what the numbers m

In [23]:
!pip install -q -U langchain -groq pydantic


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -g


In [25]:
from langchain.tools import tool
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Optional
import os
from getpass import getpass
from langchain_groq import ChatGroq

In [27]:
import os
from getpass import getpass

raw_key = getpass("Paste your NEW Groq API key: ").strip()

if not raw_key.startswith("gsk_"):
    raise ValueError("This does not look like a Groq API key. It should start with 'gsk_'.")

os.environ["GROQ_API_KEY"] = raw_key
print("Groq API Key set and verified successfully!")

Paste your NEW Groq API key: ··········
Groq API Key set and verified successfully!


In [31]:
!pip install -q groq

from groq import Groq
from getpass import getpass

new_key = getpass("Enter your new Groq key: ").strip()

client = Groq(api_key=new_key)

try:
    models = client.models.list()
    print("SUCCESS! Your API key is valid.")
    print("Available models:")
    for model in models.data[:5]:
        print(model.id)

except Exception as e:
    print("FAILED:", type(e).__name__)
    print(str(e))

Enter your new Groq key: ··········
SUCCESS! Your API key is valid.
Available models:
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
allam-2-7b


In [32]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("LLM model initialized successfully!")

LLM model initialized successfully!


In [3]:
@tool
def calculate_percentage(marks_obtained: float, total_marks: float) -> str:
    """Calculate a student's marks percentage.

    Use this when someone provides marks obtained and maximum marks.
    Example: marks_obtained=450, total_marks=500.
    """
    if total_marks <= 0:
        return "Total marks must be greater than zero."

    if marks_obtained < 0 or marks_obtained > total_marks:
        return "Marks obtained must be between zero and total marks."

    percentage = (marks_obtained / total_marks) * 100
    return f"Student percentage: {percentage:.2f}%"


@tool
def classify_performance(score: int) -> str:
    """Classify a student's final grade as Pass, Fail, or Dropout.

    Use this when the student's final grade is available on a 0–20 scale.
    Example: score=15.
    """
    if score < 0 or score > 20:
        return "Score must be between 0 and 20."

    if score == 0:
        return "Dropout"
    elif score <= 9:
        return "Fail"
    else:
        return "Pass"


print("Tools created successfully!")

Tools created successfully!


In [4]:
print(calculate_percentage.invoke({
    "marks_obtained": 450,
    "total_marks": 500
}))

print(classify_performance.invoke({
    "score": 15
}))

Student percentage: 90.00%
Pass


In [5]:
class StudentRecord(BaseModel):
    name: str = Field(description="Student's full name")
    subject: str = Field(description="Subject or course name")
    marks: float = Field(description="Marks obtained")
    attendance: float = Field(
        ge=0, le=100,
        description="Attendance percentage from 0 to 100"
    )
    date: str = Field(description="Date mentioned in the message")
    concern: Optional[str] = Field(
        default=None,
        description="Student's academic concern, if mentioned"
    )

In [12]:
structured_llm = llm.with_structured_output(StudentRecord)

In [13]:
class StudentRecord(BaseModel):
    name: str = Field(description="Student full name")
    subject: str = Field(description="Subject name")
    marks: float = Field(description="Marks obtained")
    attendance: float = Field(ge=0, le=100)
    date: str = Field(description="Date of the record")
    concern: Optional[str] = Field(default=None)

In [14]:
structured_llm = llm.with_structured_output(StudentRecord)

print("Structured output is ready!")

Structured output is ready!


In [34]:
from langchain_groq import ChatGroq

# Re-initialize the LLM and the structured engine with the updated API Key
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
structured_llm = llm.with_structured_output(StudentRecord)

message = """
My name is Ravi. I got 78 marks in Mathematics.
My attendance is 90 percent.
Date is 26 November 2026.
I need help preparing for exams.
"""

record = structured_llm.invoke(
    "Extract the student details from this message. "
    "Do not invent missing information. Message: " + message
)

print(record.model_dump())

{'name': 'Ravi', 'subject': 'Mathematics', 'marks': 78.0, 'attendance': 90.0, 'date': '26 November 2026', 'concern': None}
